In [38]:
import random
from pokerkit import *
from dotenv import load_dotenv
import json
from anthropic import Anthropic, beta_tool
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch
import re
import time
import json
import os
from datetime import datetime
from pathlib import Path

## Setup

### Memory modules

In [3]:
class MediumTermMemory:
    """
    Append-only table-scoped buffer that accumulates hand records and trend
    observations across an entire game (one session at a single table).
    Order of calls:
        new_game() called by the game engine when a new table session begins
        ingest_hand() called after each hand with the raw record from ShortTermMemory.purge_memory(); appends raw log and bumps stats
        log_trend() called by the agent after reflecting on a hand to record a human-readable trend observation (e.g. "P2 has 3-bet preflop
                    in 4 of 6 hands — likely a wide 3-bet range")
        read_digest() called by the agent at the start of each turn; returns a structured summary (stats + recent trends) rather than the
                    full raw log, keeping context window usage bounded
        read_raw() returns the full unabridged game log (for end-of-game reflection)
        close_game() seals the buffer with final chip counts and a game-level critique
        purge_memory() wipes the buffer; returns the full game record for long-term ingestion

    File layout under buffer_dir/:
        game_<id>/
            raw_log.txt every hand record, appended verbatim
            trends.txt agent trend observations, one per hand
            stats.json running numeric stats (hands played, vpip counts, etc.)
            metadata.json game-level metadata (start time, players, game_id)
    """

    def __init__(self, buffer_dir: str = "./memory/medium_term"):
        self.buffer_dir = Path(buffer_dir)
        self.buffer_dir.mkdir(parents=True, exist_ok=True)
        self._game_dir: Path | None = None
        self.game_id: str = ""
        self.hands_played: int = 0


    def new_game(self, game_id: str, players: list[str]):
        """
        Begin a new game session.
        Args:
        game_id:  unique identifier for this table session (e.g. "game_001")
        players:  list of player labels at the table, e.g. ["Hero", "P2", "P3"]
        """
        self.game_id = game_id
        self.hands_played = 0
        self._game_dir = self.buffer_dir / game_id
        self._game_dir.mkdir(parents=True, exist_ok=True)

        metadata = {
            "game_id": game_id,
            "started_at": datetime.now().isoformat(),
            "players": players,
        }
        self._write_json("metadata.json", metadata)

        #for each player:
        stats = {
            "hands_played": 0,
            "players": {p: self._empty_player_stats() for p in players},
        }
        self._write_json("stats.json", stats)

        # log games
        self._append_to("raw_log.txt",
            f"=== GAME {game_id} STARTED | {metadata['started_at']} ===\n"
            f"Players: {', '.join(players)}\n\n"
        )
        self._append_to("trends.txt",
            f"=== TREND LOG | GAME {game_id} ===\n\n"
        )

    def ingest_hand(self, action_history: list[str], reasoning: list[list], chip_changes: list[int])->None:
        """
        Alternative method to ingest hand to deprecate shortterm memory. Instead,
        read from the json and the reasoning list.
        """
        self.hands_played += 1
        #action history is a list of string
        for item in action_history:
            self._append_to("raw_log.txt", f'{item}\n')

        self._append_to("raw_log.txt", '\nReasonings:\n')
        streets = ["Preflop", "Flop", "Turn", "River"]
        for reasoning_on_street, streetname in zip(reasoning, streets):
            self._append_to("rawlog.txt", f'{streetname}: {reasoning_on_street}')
        stats = self._read_json("stats.json")
        stats["hands_played"] = self.hands_played
        stats["chip_changes"] = chip_changes

        self._write_json("stats.json", stats)



    def log_trend(self, observation: str, hand_number: int | None = None) -> None:
        """
        Record a free-prose trend observation after hand-level reflection.

        Args:
        observation: the agent's natural-language finding, e.g.:
            "P2 has continuation-bet the flop in 3/4 opportunities — treat flop
             c-bets from P2 as weak until proven otherwise."
        hand_number: if provided, prefixes the entry for traceability.
        """
        hand_ref = f"[After hand #{hand_number}]" if hand_number is not None else f"[After hand #{self.hands_played}]"
        entry = f"{hand_ref}\n{observation.strip()}\n\n"
        self._append_to("trends.txt", entry)


    def read_digest(self, recent_trends: int = 5):
        """
        Return a bounded summary suitable for injection into the agent's context
        at the start of each turn.

        Includes:
          - Running numeric stats (frequencies derived from stats.json)
          - The N most recent trend observations
        unlike read_raw(), this is a summary

        Args:
        recent_trends: how many of the latest trend entries to include (default 5).
        """
        if not self._game_dir:
            return "[Medium term memory is empty — no active game]"

        stats = self._read_json("stats.json")
        n = max(stats.get("hands_played", 0), 1) #div/0 error sentinel

        lines: list[str] = [f"=== LIVE TABLE DIGEST | Game {self.game_id} | {n} hands played ===\n"]

        for player, bucket in stats.get("players", {}).items():
            vpip_pct = 100 * bucket.get("vpip", 0)/n
            pfr_pct = 100 * bucket.get("preflop_raise", 0)/n
            pf3b_pct = 100 * bucket.get("preflop_3bet", 0)/n
            cbet_pct = 100 * bucket.get("continuation_bet", 0)/n
            wtsd_pct = 100 * bucket.get("went_to_showdown", 0)/n
            net = bucket.get("net_chips", 0)
            lines.append(
                f"  {player}: VPIP={vpip_pct:.0f}% | PFR={pfr_pct:.0f}% | "
                f"3B={pf3b_pct:.0f}% | CBet={cbet_pct:.0f}% | "
                f"WTSD={wtsd_pct:.0f}% | Net={net:+d} chips"
            )
        lines.append("")

        trends_text = self._read_text("trends.txt")
        #sptit on \n\n means we split on blank single lines.
        entries = [e.strip() for e in trends_text.split("\n\n") if e.strip() and not e.strip().startswith("=== TREND LOG")]
        if entries:
            lines.append(f"--- Recent Observations (last {recent_trends}) ---")
            for entry in entries[-recent_trends:]:
                lines.append(entry)
        else: lines.append("--- No trend observations recorded yet ---")

        return "\n".join(lines)

    def read_raw(self):
        """Return the full unabridged game log. Used for end-of-game reflection."""
        if not self._game_dir:
            return "[Medium term memory is empty — no active game]"
        return self._read_text("raw_log.txt")


    def close_game(self, final_stacks: dict, game_critique: str = ""):
        """
        Seals the buffer with final results and an optional game-level critique.
        Args:
        final_stacks: e.g. {"Hero": 620, "P2": 430, "P3": 450}
        game_critique: agent's high-level reflection on the session
        """
        stacks_str = " | ".join(f"{p}: {c}" for p, c in final_stacks.items())
        footer = (
            f"\n=== GAME {self.game_id} CLOSED | {datetime.now().isoformat()} ===\n"
            f"Final stacks: {stacks_str}\n"
            f"Hands played: {self.hands_played}\n"
            f"Game critique: {game_critique.strip() or 'pending'}\n"
            f"{'=' * 60}\n"
        )
        self._append_to("raw_log.txt", footer)
        self._append_to("trends.txt", footer)

    def purge_memory(self):
        """
        Return the full game record (raw log + trends concatenated) and wipe
        the game directory from disk.
        The returned string is what gets handed to the long-term reflection
        pipeline for player-profile updates.
        """
        if not self._game_dir:
            return "[Nothing to purge — no active game]"

        raw = self._read_text("raw_log.txt")
        trends = self._read_text("trends.txt")
        combined = f"{raw}\n\n{'=' * 60}\n\n{trends}"

        # Wipe game directory.
        for child in self._game_dir.iterdir():
            child.unlink()
        self._game_dir.rmdir()

        self._game_dir = None
        self.game_id = ""
        self.hands_played = 0

        return combined



  #internal class methods below

    @staticmethod
    def _empty_player_stats():
        return {
            "vpip": 0,
            "preflop_raise": 0,
            "preflop_3bet": 0,
            "continuation_bet": 0,
            "went_to_showdown": 0,
            "won_hand": 0,
            "net_chips": 0,
        }

    def _path(self, filename: str) -> Path:
        assert self._game_dir is not None, "Call new_game() before accessing memory."
        return self._game_dir / filename

    def _append_to(self, filename: str, text: str) -> None:
        with open(self._path(filename), "a", encoding="utf-8") as f:
            f.write(text)

    def _read_text(self, filename: str) -> str:
        p = self._path(filename)
        if not p.exists():
            return ""
        return p.read_text(encoding="utf-8")

    def _read_json(self, filename: str) -> dict:
        p = self._path(filename)
        if not p.exists():
            return {}
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            return {}

    def _write_json(self, filename: str, data: dict) -> None:
        with open(self._path(filename), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

class LongTermMemory:
    """
    Long term memory system inspired by mongodb.

    Each player gets one text file including us (self supervised strategy), + one index.json file
    that indexes store for quick lookup.

    Profile text files are weakly structured, but agent has free will in managing how
    it is written.

    Use order:
        new_session() called before a game begins; loads existing profiles from disk into memory
        lookup_player() called by the agent at turn-start to retrieve an opponent's full profile text
        read_self_profile() called by the agent to read its own strategic notes
        list_known_players() called by the agent to enumerate known opponents optionally filtered by behavioural tags
        log_player_update() called by the reflection pipeline after each game to append new observations to an opponent's profile
        log_self_update() called by the reflection pipeline after each game to append new self-notes
        close_session() closes the session; rebuilds the index from disk
        rebuild_index() rebuiilds index.json from .txt files

    File layout under root/:
        players/
            <player_id>.txt full prose profile, one file per known opponent
        self.txt gent's own strategic self-notes
        index.json  queryable metadata sidecar (tags, last_seen, etc.)
    """

    INDEX_FILENAME = "index.json"
    PLAYERS_DIRNAME = "players"
    SELF_FILENAME = "self.txt"

    _NO_PROFILE = "No prior profile for this player."
    _SELF_SCAFFOLD = (
        "# Self Profile\n"
        "## Last updated: never\n"
        "## Games played: 0\n\n"
        "## Current strategy\n"
        "(no strategy notes yet)\n\n"
        "## Known leaks\n"
        "(none identified yet)\n\n"
        "## Lessons learned\n"
        "(none yet)\n\n"
        "## Open questions\n"
        "(none yet)\n"
    )


#touch these regexes at own risk
    _HEADER_RE = re.compile(r"^##\s*([^:]+?)\s*:\s*(.*?)\s*$")
    _SECTION_RE = re.compile(r"^##\s+([^:]+?)\s*$")

    def __init__(self, root: str = "./memory/long_term"):
        self.root = Path(root)
        self.players_dir = self.root / self.PLAYERS_DIRNAME
        self.index_path = self.root / self.INDEX_FILENAME
        self.self_path = self.root / self.SELF_FILENAME

        self.root.mkdir(parents=True, exist_ok=True)
        self.players_dir.mkdir(parents=True, exist_ok=True)

        self._index: dict[str, dict] = self._load_index()
        self._session_active: bool = False


    def new_session(self, game_id: str):
        """
        Loads index from disk so the agent has access to all previously observed player profiles.
        Args:
        game_id: identifier for the current game, e.g. "game_003"
        """
        self._index = self._load_index()
        self._session_active = True
        self._session_game_id = game_id

    def close_session(self):
        """
        Rebuilds the index from .txt files on disk
        to catch any changes by log_player_update() writes.
        """
        assert self._session_active, "Call new_session() before close_session()."
        self.rebuild_index()
        self._session_active = False


    def lookup_player(self, player_id: str):
        """
        Returns the full profile text for player_id.
        Returns a _NO_PROFILE constant if not found.
        """
        path = self._player_path(player_id)
        if not path.exists():
            return self._NO_PROFILE
        return path.read_text(encoding="utf-8")

    def read_self_profile(self):
        """
        Returns the agent's own strategic self-profile. Creates the file with header
        first if does not exist and returns that.
        """
        if not self.self_path.exists():
            self.self_path.write_text(self._SELF_SCAFFOLD, encoding="utf-8")
        return self.self_path.read_text(encoding="utf-8")

    def list_known_players(self, filter_tags: list[str] | None = None):
        """
        Return index entries for all known players, optionally filtered by tag.
        A player is included if at least one of their tags appears in filter_tags.
        Tag matching is case-insensitive. Passing an empty list filters to nothing;
        passing None returns all.

        Returns a list of dictionaries:
        Each returned dict has the shape:
            {
                "id":str,
                "tags": list[str],
                "last_seen" str,
                "games_observed":int,
                "summary":str,
            }

        Results are ordered most-recently-seen first, then alphabetically by id.
        """
        if filter_tags is None:
            wanted: set[str] | None = None
        else:
            wanted = {t.strip().lower() for t in filter_tags if t.strip()}

        results: list[dict] = []
        for player_id, entry in self._index.items():
            if wanted is not None:
                player_tags = {t.lower() for t in entry.get("tags", [])}
                if not (wanted & player_tags):
                    continue
            results.append({"id": player_id, **entry})

        results.sort(
            key=lambda r: (r.get("last_seen", ""), r["id"]),
            reverse=True,
        )
        return results


    def log_player_update(self, player_id: str, observation: str, tags: list[str] | None = None, summary: str | None = None):
        """
        Appends a new observation to an opponent's profile after end-of-game
        reflection. Creates if it does not exist yet.

        Args:
        player_id: opponent label, e.g. "P2"
        observation: free-prose finding, e.g. "Consistently 3-bets light from the BTN; likely a polarised 3-bet range."
        tags:        behavioural tags to attach or refresh, e.g. ["aggressive", "bluff-heavy"]. Merged with any existing tags in the index.
        summary:     short snapshot for the index. When provided, replaces the profile's Summary section. Left unchanged if omitted.
        """
        path = self._player_path(player_id)
        timestamp = datetime.now().isoformat()
        game_ref = getattr(self, "_session_game_id", "unknown")


        #if not found create.
        if not path.exists():
            header = (
                f"# Player Profile: {player_id}\n"
                f"## Tags: {', '.join(tags or [])}\n"
                f"## Last seen: {timestamp}\n"
                f"## Games observed: 1\n\n"
                f"## Summary\n"
                f"(no summary yet — see observations below)\n\n"
                f"## Observations\n"
            )
            self._write(path, header, mode="w")
            games_observed = 1
        else:
            games_observed = self._index.get(player_id, {}).get("games_observed", 0) + 1

        #write
        entry = (f"[{timestamp} | {game_ref}]\n" f"{observation.strip()}\n\n")
        self._write(path, entry, mode="a")
        self._patch_headers(path, {
            "last seen": timestamp,
            "games observed": str(games_observed),
            "tags": ", ".join(self._merge_tags(self._index.get(player_id, {}).get("tags", []), tags or []))})
        if summary is not None:
            self._patch_section(path, "Summary", summary)
        self._index[player_id] = self._parse_profile(path.read_text(encoding="utf-8"))
        self._save_index()

    def log_self_update(self, observation: str, section: str = "Lessons learned"):
        """
        Append a new self-note after end-of-game reflection. Scaffolds
        self.txt if it does not exist yet.

        Args:
        observation: free-prose strategic note, e.g. "Leaking chips by calling too wide on the river vs tight opponents."
        section:     which section header to append under. Defaults to "Lessons learned". Must already exist in the file.
        """
        _ = self.read_self_profile()
        timestamp = datetime.now().isoformat()
        game_ref = getattr(self, "_session_game_id", "unknown")
        entry = f"[{timestamp} | {game_ref}] {observation.strip()}\n"

        text = self.self_path.read_text(encoding="utf-8")
        target = f"## {section}"
        if target not in text:
            text += f"\n## {section}\n{entry}"
        else:
            lines = text.splitlines(keepends=True)
            out: list[str] = []
            for line in lines:
                out.append(line)
                if line.strip() == target:
                    out.append(entry)
            text = "".join(out)

        self.self_path.write_text(text, encoding="utf-8")
        self._patch_headers(self.self_path, {"last updated": timestamp})


    def rebuild_index(self):
        """
        Helper to rebuild index.json from the .txt files on disk.
        """
        new_index: dict[str, dict] = {}
        for txt_path in self.players_dir.glob("*.txt"):
            player_id = txt_path.stem
            try:
                text = txt_path.read_text(encoding="utf-8")
            except OSError:
                continue
            new_index[player_id] = self._parse_profile(text)
        self._index = new_index
        self._save_index()



#Helper functions
    def _player_path(self, player_id: str):
        if "/" in player_id or "\\" in player_id or ".." in player_id:
            raise ValueError(f"Invalid player_id: {player_id!r}")
        return self.players_dir / f"{player_id}.txt"

    def _write(self, path: Path, text: str, mode: str = "a"):
        with open(path, mode, encoding="utf-8") as f:
            f.write(text)

    def _load_index(self):
        if not self.index_path.exists():
            return {}
        try:
            raw = json.loads(self.index_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            return {}
        return raw if isinstance(raw, dict) else {}

    def _save_index(self):
        self.index_path.write_text(
            json.dumps(self._index, indent=2, sort_keys=True),
            encoding="utf-8",
        )

    def _parse_profile(self, text: str):
        entry = {"tags": [], "last_seen": "", "games_observed": 0, "summary": ""}
        lines = text.splitlines()
        i = 0
        while i < len(lines):
            line = lines[i]

            m = self._HEADER_RE.match(line)
            if m:
                key = m.group(1).strip().lower()
                val = m.group(2).strip()
                if key == "tags":
                    entry["tags"] = [t.strip() for t in val.split(",") if t.strip()]
                elif key == "last seen":
                    entry["last_seen"] = val
                elif key == "games observed":
                    try:
                        entry["games_observed"] = int(val)
                    except ValueError:
                        pass
                i += 1
                continue

            m = self._SECTION_RE.match(line)
            if m and m.group(1).strip().lower() == "summary":
                j = i + 1
                buf: list[str] = []
                while j < len(lines) and not lines[j].lstrip().startswith("##"):
                    buf.append(lines[j])
                    j += 1
                entry["summary"] = self._first_paragraph(buf)
                i = j
                continue

            i += 1

        return entry

    @staticmethod
    def _first_paragraph(lines: list[str]):
        paragraph: list[str] = []
        for line in lines:
            stripped = line.strip()
            if not stripped:
                if paragraph:
                    break
                continue
            paragraph.append(stripped)
        return " ".join(paragraph)

    @staticmethod
    def _merge_tags(existing: list[str], new: list[str]):
        seen: set[str] = set()
        merged: list[str] = []
        for tag in existing + new:
            if tag.lower() not in seen:
                seen.add(tag.lower())
                merged.append(tag)
        return merged

    def _patch_headers(self, path: Path, updates: dict[str, str]):
        text = path.read_text(encoding="utf-8")
        lines = text.splitlines(keepends=True)
        out: list[str] = []
        for line in lines:
            m = self._HEADER_RE.match(line.rstrip("\n"))
            if m and m.group(1).strip().lower() in updates:
                key_display = m.group(1)
                new_val = updates[m.group(1).strip().lower()]
                line = f"## {key_display}: {new_val}\n"
            out.append(line)
        path.write_text("".join(out), encoding="utf-8")

    def _patch_section(self, path: Path, section: str, body: str):
        text = path.read_text(encoding="utf-8")
        lines = text.splitlines(keepends=True)
        out: list[str] = []
        target = f"## {section}"
        i = 0
        while i < len(lines):
            out.append(lines[i])
            if lines[i].rstrip("\n").strip() == target:
                out.append(f"{body.strip()}\n")
                out.append("\n")
                i += 1
                while i < len(lines) and not lines[i].lstrip().startswith("##"):
                    i += 1
                continue
            i += 1
        path.write_text("".join(out), encoding="utf-8")

### LLM loader

In [4]:
load_dotenv()

client = Anthropic()

### LLM integration functions

See comment for instructions on how to enable/disable equity calculator for ablation results.

In [5]:
PLAYER_LABEL_MAP = dict(zip([i for i in range(1,9)], [f"P{i+1}" for i in range(1,9)])) #our opponents, for example seat 1 has P2, logged as P2.txt


def build_system_prompt(ltm: LongTermMemory, active_player_ids: list[str]) -> str:
    """Construct the system prompt, injecting LTM profiles for known opponents."""

    base = (
        f"You are a poker agent playing {len(active_player_ids) + 1}-handed Texas Hold'em No-Limit.\n"
        "Blinds 1000/2000. You are Hero. Think about pot odds, position, hand strength, stack depth.\n"
        "Use the take_action tool to act.\n"
        "Use self-notes and opponent notes to inform your decisions."
    )

    full_self = ltm.read_self_profile()
    self_lines = full_self.strip().splitlines()
    self_snippet = "\n".join(self_lines[-20:])
    ltm_section = f"\n\n--- SELF NOTES (recent) ---\n{self_snippet}\n---"

    known = {p["id"]: p for p in ltm.list_known_players()}
    opponent_parts = []
    for pid in active_player_ids:
        if pid in known:
            entry = known[pid]
            games = entry.get("games_observed", 0)
            # Full profile only if seen recently (≥2 games), else just the summary line
            if games >= 2:
                profile = ltm.lookup_player(pid)
                # Trim to last 40 lines to avoid runaway growth
                lines = profile.strip().splitlines()
                trimmed = "\n".join(lines[:8] + ["..."] + lines[-15:]) if len(lines) > 25 else profile
                opponent_parts.append(f"\n--- {pid} (seen {games}x) ---\n{trimmed}\n---")
            else:
                tags = ", ".join(entry.get("tags", [])) or "unknown"
                summary = entry.get("summary", "")[:120]
                opponent_parts.append(f"\n--- {pid}: tags=[{tags}] summary={summary} ---")
        else:
            opponent_parts.append(f"\n--- {pid}: no prior data ---")

    return base + ltm_section + "".join(opponent_parts)


def extract_tags_from_reasoning(reasoning_texts: list[str]) -> dict[str, list[str]]:
    tag_map: dict[str, list[str]] = {}

    for text in reasoning_texts:
        if not text:
            continue
        words = text.split()
        for i, word in enumerate(words):
            player_label = word.strip(",:").upper()
            if not (player_label.startswith("P") and player_label[1:].isdigit()):
                continue
            if player_label[1] not in "123456789": #! we only support opponent names P1 through P9
                continue
            #search for rating
            lookahead = words[i+1 : i+11]
            for chunk in lookahead:
                if "/10" not in chunk:
                    continue
                score_part = chunk.split("/10")[0].strip()
                if not score_part.isdigit():
                    break
                score = int(score_part)
                if score >= 7:
                    tag = "aggressive"
                elif score <= 3:
                    tag = "passive"
                else:
                    tag = "balanced"
                tag_map.setdefault(player_label, []).append(tag)
                break
    return {pid: [Counter(tags).most_common(1)[0][0]] for pid, tags in tag_map.items()}

def extract_player_summary(eval_text: str, pid: str) -> str:
    paragraphs = [para.strip() for para in eval_text.split("\n\n") if para.strip()]
    for paragraph in paragraphs:
        header = paragraph.split(":", 1)[0]
        if pid in re.findall(r"P\d", header):
            return paragraph
    return eval_text

def run_post_hand_reflection(ltm: LongTermMemory, eval_text: str, active_player_ids: list[str], reasonings: list, payoffs: tuple) -> None:
    """
    Called once after each hand to:
    1. Write per-opponent observations to LTM.
    2. Append a self-note about how Hero played.
    """
    if not eval_text: return

    reasoning_texts = [r[2] for r in reasonings if len(r) >= 3]
    tag_map = extract_tags_from_reasoning(reasoning_texts)
    for pid in active_player_ids:
        tags = tag_map.get(pid, [])
        ltm.log_player_update(player_id=pid, observation=eval_text, tags=tags, summary=extract_player_summary(eval_text, pid))
    hero_payoff = payoffs[0] if payoffs else 0
    outcome = "won" if hero_payoff > 0 else ("lost" if hero_payoff < 0 else "broke even")
    hero_reasoning = " | ".join(f"{reasoning[0]} {reasoning[1] if reasoning[1] else ''}: {reasoning[2]}"
        for reasoning in reasonings if len(reasoning) >= 3)
    self_note = (
        f"Hand outcome: {outcome} ({hero_payoff:+,} chips). "
        f"Opponent eval: {eval_text[:300]}. "
        f"Hero reasoning summary: {hero_reasoning[:400]}."
    )
    ltm.log_self_update(observation=self_note, section="Lessons learned")


@beta_tool
def take_action(action: str, reasoning: str, amount: float = 0) -> str:
    """Submit your poker action for this turn and give a maximum 3 sentence explanation for why.

    Args:
        action: One of fold, check, call, raise
        amount: Raise amount in chips. Required if action is raise.
        reasoning: Reasoning for why you took a poker action. Maximum 3 sentences. Rate how aggressive each opponent's actions are on a 1-10 scale, but only if you made it to the flop or later.
    """
    return json.dumps({"status": "accepted", "action": action, "amount": amount, 'reasoning': reasoning})

@beta_tool
def get_equity_strength(hole_cards: str, board_cards: str, active_players: int) -> str:
    """Calculates the mathematical win probability (equity) of your current hand using Monte Carlo simulations.
    Call this tool when you are facing a difficult decision, a large bet, or are unsure of your exact hand strength.

    Args:
        hole_cards: Your two hole cards as a string (e.g., 'AsKc').
        board_cards: The visible board cards as a string (e.g., 'Td9h2c'). Leave empty '' for preflop.
        active_players: Total number of players dealt into the simulation.
    """

    print(f"\n   [TOOL TRIGGERED] Calculating equity for {hole_cards} | Board: {board_cards} | Players: {active_players}")

    board_input = tuple(Card.parse(board_cards)) if board_cards else ()

    calculated_equity = calculate_hand_strength(
        active_players,
        parse_range(hole_cards),
        board_input,
        active_players,
        5,
        Deck.STANDARD,
        (StandardHighHand,),
        sample_count=500
    )

    win_percentage = round(calculated_equity * 100, 1)

    return json.dumps({
        "status": "success",
        "equity": calculated_equity,
        "message": f"You have a {win_percentage}% chance of winning."
    })


def run_turn(state_json, ltm: LongTermMemory, active_player_ids: list[str]):
    system_prompt = build_system_prompt(ltm, active_player_ids)

    messages = [{
        "role": "user",
        "content": f"""It's your turn to act. Here is the current game state:

    ```json
    {state_json}
    ```

    Decide your action."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        # tools=[take_action], <----------------------- Disable next line and enable this line to test the bot without the equity calculator
        tools = [take_action, get_equity_strength],
        system=system_prompt,
        messages=messages
    )

    agent_action = None
    tokens = ""

    for message in runner:
        tokens = f"Input: {message.usage.input_tokens}, Output: {message.usage.output_tokens}"
        for block in message.content:
            if block.type == "tool_use" and block.name == "take_action":
                agent_action = block.input

    return agent_action, tokens

def street_converter(index):
    if index == 0:
        return 'preflop'
    elif index == 1:
        return 'flop'
    elif index == 2:
        return 'turn'
    elif index == 3:
        return 'river'

def get_visible_operations(state):
    visible = []
    showdown = sum(1 for p in state.statuses if p) > 1
    winners = set()
    for op in state.operations:
        if isinstance(op, ChipsPushing):
            winners = {i for i, amt in enumerate(op.amounts) if amt > 0}

    for op in state.operations:
        if isinstance(op, HoleDealing):
            continue
        elif isinstance(op, CardBurning):
            continue
        elif isinstance(op, HoleCardsShowingOrMucking):
            if showdown or op.player_index in winners:
                visible.append(op)
        else:
            visible.append(op)
    return visible

judge_system_prompt = """You are an evaluating agent for players playing Texas Hold-em. Your job is to evaluate and provide useful information about the other non-agent players to aid Player 1 in defeating them. Do not evaluate Player 1 (Hero), focus on the other players, denoted P2 and up. Remember that index 0 is the agent, index 1 is Player 2 and so on. Use the generate_observation tool to submit your decision."""

@beta_tool
def generate_observation(evaluation: str) -> str:
    """Evaluate the action history of the game in the previous poker hand.

    Args:
        evaluation: The evaluation of how the other players played. Be specific as to any mistakes or good moves each non-Hero player made.
    """
    return json.dumps({"status": "accepted", "evaluation": evaluation})


def run_observation_generator(action_history):
    # eval_json = {'action_history': action_history}
    trimmed = action_history[-30:]
    eval_json = {'action_history': trimmed}
    messages = [{
        "role": "user",
        "content": f"""Here is the action history of the game.

    ```json
    {eval_json}
    ```

    Generate your observations."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[generate_observation],
        system=judge_system_prompt,
        messages=messages
    )

    agent_action = None

    for message in runner:
        for block in message.content:
            if block.type == "tool_use" and block.name == "generate_observation":
                agent_action = block.input["evaluation"]

    return agent_action


def _position_name(seat):
    if seat == 0:
        return "SB"
    if seat == 1:
        return "BB"
    return "other"


def humanize_action_history(ops, active_indices, players):
    history = []
    for op in ops:
        s = str(op)
        def repl(m):
            seat = int(m.group(1))
            return f'player={players[active_indices[seat]]}'
        s = re.sub(r'player_index=(\d+)', repl, s)
        history.append(s)
    return history


### Opponents

In [ ]:
def ev_based_strategy(state, player_index, ev_pot_threshold=0.5, num_simulations=500):
    """
    Bets/raises when equity > ev_pot_threshold, calls when pot odds justify it,
    else folds.
    """
    hole = state.hole_cards[player_index]

    if len(hole) < 2:
        if state.can_check_or_call():
            state.check_or_call()
        elif state.can_fold():
            state.fold()
        return

    active_players = sum(1 for s in state.statuses if s)
    board_input = tuple(c for street in state.board_cards for c in street)
    hole_range = {frozenset(hole)}

    equity = calculate_hand_strength(
        active_players,
        hole_range,
        board_input,
        2,
        5,
        Deck.STANDARD,
        (StandardHighHand,),
        sample_count=num_simulations
    )

    pot = state.total_pot_amount or 1

    if equity > ev_pot_threshold and state.can_complete_bet_or_raise_to():
        lo = state.min_completion_betting_or_raising_to_amount
        hi = state.max_completion_betting_or_raising_to_amount
        scale = min(equity, 1.0)
        target = int(lo + scale * (hi - lo))
        target = max(lo, min(target, hi))
        state.complete_bet_or_raise_to(target)
    elif state.can_check_or_call():
        call_amount = state.checking_or_calling_amount or 0
        if call_amount == 0 or equity > call_amount / (pot + call_amount):
            state.check_or_call()
        elif state.can_fold():
            state.fold()
        else:
            state.check_or_call()
    elif state.can_fold():
        state.fold()
    else:
        state.check_or_call()



def pocket_pair_strategy(state, player_index):
    hole = state.hole_cards[player_index]
    is_pair = len(hole) >= 2 and hole[0].rank == hole[1].rank
    strong_pair_ranks = {"A", "K", "Q", "J", "T", "9", "8", "7"}
    is_strong_pair = is_pair and hole[0].rank in strong_pair_ranks
    if is_strong_pair and state.can_complete_bet_or_raise_to():
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif is_pair and state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()
    else:
        state.check_or_call()

def punter_strategy(state, player_index):
    punter_is_punting_this_hand = random.random() < 0.1
    if not punter_is_punting_this_hand:
        if state.can_fold():
            state.fold()
        else:
            state.check_or_call()
        return
    if state.can_complete_bet_or_raise_to():
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()

def tight_passive_strategy(state, player_index):
    r = random.random()
    if r < 0.9 and state.can_fold():
        state.fold()
    elif state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()

def random_raiser(state, player_index):
    action_rand = random.random()
    if action_rand < 0.1 and state.can_complete_bet_or_raise_to():
    # 10% chance to raise to 3x the big blind
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount

    # Raise to 2x the minimum required to keep it moving
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif action_rand < 0.2 and state.can_fold():
        state.fold()
    else:
        state.check_or_call()

Edit the below cell to change what roles each opponent plays. Roles can be repeated. To remove a player simply delete that entry, but there can be no gaps in the numerical order (i.e. a 5-player game must have index 1, 2, 3, 4, and 5)

In [6]:
opponent_strategies = {
    1: pocket_pair_strategy,
    2: punter_strategy,
    3: tight_passive_strategy,
    4: random_raiser,
    5: ev_based_strategy,
    6: lambda st, id: ev_based_strategy(st, id, ev_pot_threshold=0.7)} # This is a more conservative version of player 5

### Plotter function

In [6]:
def plotter(stacks, game_number, players, colors, folder):
    df = pd.DataFrame(stacks, columns=players)
    hand_number = len(df)
    win_pct = (df.Hero.diff() > 0).sum() / df.Hero.diff().notna().sum()
    hands = np.arange(1, hand_number+1, 1)
    plt.figure(figsize=[16, 9])
    if colors is None:
        colors = ['red', 'orange', 'green', 'blue', 'indigo', 'firebrick', 'gold']
    for player, color in zip(players, colors):
        plt.plot(hands, df[player], color=color, label=player)
    plt.xlim(1, hand_number)
    plt.xlabel('Hand #')
    plt.ylabel('Chip Count')
    handles, labels = plt.gca().get_legend_handles_labels()
    handles.append(Patch(facecolor="none", edgecolor="none"))
    labels.append(f"Win %: {round(win_pct*100, 2)}%")
    plt.legend(handles=handles, labels=labels, prop={'size': 15})
    plt.title(f'Game Number {game_number}')
    if folder is not None:
        plt.savefig(f'{folder}/run_{game_number}.png', bbox_inches='tight')
    plt.show()
    return stacks[-1], win_pct*100

### Game loop function

In [8]:
def run_game(game_id, players, player_stacks, sb=1000, bb=2000, min_bet=2000, opponent_strategies=opponent_strategies):
    med = MediumTermMemory()
    med.new_game(game_id=game_id, players=players)
    ltm = LongTermMemory()
    ltm.new_session(game_id=game_id)

    # Initial Setup
    hand_count = 0
    stacks_per_hand = []
    button = len(player_stacks) - 1
    # The Global Game Loop: Runs until only one player has chips, and hero is alive, for a max of 30 hands per game
    while len([s for s in player_stacks if s > 0]) > 1 and hand_count < 30 and player_stacks[0] > 0:
        time.sleep(1)
        reasonings = []
        active_indices = [i for i, stack in enumerate(player_stacks) if stack > 0]

        if len(active_indices) < 2:
            break

        sb_global = next((i for i in active_indices if i > button), active_indices[0])
        pivot = active_indices.index(sb_global)
        active_indices = active_indices[pivot:] + active_indices[:pivot]

        # Map engine seat indices → human player IDs for LTM lookups
        # Seat 0 = Hero; seat 1+ = opponents in position order
        active_player_ids = [
            PLAYER_LABEL_MAP[i] for i in active_indices if i in PLAYER_LABEL_MAP
        ]

        current_stacks = tuple(player_stacks[i] for i in active_indices)
        hand_count += 1
        print(f"\n--- STARTING HAND {hand_count} ---")
        print(f'Button: {button}', active_indices)

        state = NoLimitTexasHoldem.create_state(
            (
                Automation.ANTE_POSTING,
                Automation.BET_COLLECTION,
                Automation.BLIND_OR_STRADDLE_POSTING,
                Automation.CARD_BURNING,
                Automation.HOLE_DEALING,
                Automation.BOARD_DEALING,
                Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
                Automation.HAND_KILLING,
                Automation.CHIPS_PUSHING,
                Automation.CHIPS_PULLING,
            ),
            True, 0, (sb, bb), min_bet, tuple(current_stacks), len(active_indices)
        )

        # Per-Turn Decision Loop
        while state.status:
            if state.can_deal_hole():
                state.deal_hole()
            elif state.can_deal_board():
                state.deal_board()

            elif state.actor_index is not None:
                global_player_index = active_indices[state.actor_index]
                res = get_visible_operations(state)
                obs = {
                    "your_index": state.actor_index,
                    "pot": state.total_pot_amount,
                    "position": _position_name(state.actor_index),
                    "board": state.board_cards,
                    "hole": state.hole_cards[state.actor_index],
                    "stacks": state.stacks,
                    "bets": state.bets,
                    "street": street_converter(state.street_index),
                    "can_fold?": state.can_fold(),
                    "can_check_or_call?": state.can_check_or_call(),
                    "can_raise?": state.can_complete_bet_or_raise_to(),
                    "min_raise": state.min_completion_betting_or_raising_to_amount,
                    "max_raise": state.max_completion_betting_or_raising_to_amount,
                    "how_much_to_call": state.checking_or_calling_amount,
                    "action_history": res
                }

                if global_player_index == 0: #hero/agent
                    action, tokens = run_turn(obs, ltm, active_player_ids)

                    if action is None:
                        print("!! run_turn returned None")
                        print(f"   obs: {obs}")
                        print(f"   hand {hand_count}, street {state.street_index}")
                        raise RuntimeError("run_turn returned None — inspect logs")

                    if action['action'] in ['check', 'call']:
                        state.check_or_call()
                    elif action['action'] == 'raise':
                        raw = action.get('amount', 0)
                        low = state.min_completion_betting_or_raising_to_amount
                        high = state.max_completion_betting_or_raising_to_amount
                        clamped = max(low, min(raw, high))
                        if clamped != raw:
                            print(f"!! clamped raise: {raw} -> {clamped} (range [{low}, {high}])")
                        state.complete_bet_or_raise_to(clamped)
                    elif action['action'] == 'fold':
                        state.fold()
                    reasonings.append([action['action'], action.get('amount', 0), action.get('reasoning', '')])

                else:  # Random bots
                    opponent_strategies[global_player_index](state, state.actor_index)
            else:
                break

        payoffs_by_player = [0] * len(player_stacks)
        for seat, global_idx in enumerate(active_indices):
            payoffs_by_player[global_idx] = int(state.payoffs[seat])
        ops = get_visible_operations(state)
        history = humanize_action_history(ops, active_indices, players)
        med.ingest_hand(action_history=history, reasoning=reasonings,
            chip_changes=payoffs_by_player)
        eval_text = run_observation_generator(history)
        med.log_trend(eval_text)

        run_post_hand_reflection(ltm=ltm, eval_text=eval_text, active_player_ids=active_player_ids, reasonings=reasonings, payoffs=payoffs_by_player)

        for g in range(len(player_stacks)):
            player_stacks[g] += payoffs_by_player[g]
        stacks_per_hand.append(player_stacks.copy())

        print(f"Hand {hand_count} Results: {payoffs_by_player}")
        print(f"New Stacks: {player_stacks}")

        button = (button + 1) % len(player_stacks)
        while player_stacks[button] <= 0:
            button = (button + 1) % len(player_stacks)
    ltm.close_session() #flush to disk

    print(f"\nGAME OVER after {hand_count} hands!")
    print(f"Final Winner Stacks: {player_stacks}")
    print(f"\nLTM index now contains: {list(ltm._index.keys())}")
    return stacks_per_hand

## Experiments and plotting

### Main loop

In [24]:
res = []

The below loop runs a standard 10 game loop capped at 30 hands with all players. The second function argument is the player list, and must match the number of opponents in the opponent strategies dictionary. The third argument must also be the same length and determines the number of starting chips per hand.

In [ ]:
for i in range(0,10):
    out = run_game('00'+str(i), ['Hero', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7'], [20_000, 20_000, 20_000, 20_000, 20_000, 20_000, 20_000])
    res.append(out)
    df = pd.DataFrame(out)
    df.to_csv(f'output_game_00{i}.csv')

### Plotting

In [ ]:
finals = []
for i in range(0, len(res)):
    fin_counts, win_pct = plotter(res[i], i+1, ['Hero', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7'], colors=None, folder='ablation_results')
    fin_counts.append(win_pct)
    finals.append(fin_counts)

In [ ]:
fin_res = pd.DataFrame(finals, columns=['Agent', 'Pocket Pair', 'Punter', 'Passive', 'Random', 'EV 0.5', 'EV 0.7', 'Agent Win %'])
fin_res